# Smoke test (yolo11n, 3 epochs, imgsz=320)

End-to-end sanity check that takes ~5 min on a T4. Use this before kicking off a real
training run. Verifies: Drive mount, repo clone, dataset unzip + sha256, training
loop, run_meta.json write, atomic copy to Drive, and test-split eval.

In [ ]:
REPO_URL    = "https://github.com/tahmid013/yolo.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os, shutil, subprocess, sys
if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')
subprocess.run(
    ['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, '/content/code'],
    check=True,
)
os.chdir('/content/code')
if '/content/code' not in sys.path:
    sys.path.insert(0, '/content/code')
# purge any stale `pipeline` cache from previous kernel state
for mod in [m for m in list(sys.modules) if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[mod]

!pip install -q -r requirements.txt

In [ ]:
from pipeline.dataset import ensure_dataset
from pipeline import paths
data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)

In [ ]:
from pipeline.train import run as train_run
run_dir = train_run(
    model='yolo11n',
    config='configs/smoke.yaml',
    drive_runs_dir=paths.RUNS_DIR,
    local_runs_dir=paths.LOCAL_RUNS,
    data_yaml=data_yaml,
    dataset_meta_path=paths.dataset_meta(DATASET_VERSION),
    base_config='configs/base.yaml',
)
print('Smoke run on Drive:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_run(run_dir=run_dir, data_yaml=data_yaml, drive_runs_dir=paths.RUNS_DIR)